<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_5_Bedragerier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 Maskininlärning – Lektion 5: Kreditkortsbedrägerier och obalanserad data

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 40 minuter  
**Mål:** Förstå problemet med obalanserad data och varför noggrannhet (Accuracy) kan vara ett missvisande mått

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🌍 Del 1 – Från blommor till verkligheten

Vi har jobbat med att klassificera iris-blommor – det var ett bra sätt att lära sig!  
Men nu tar vi ett steg mot ett **riktigt problem** som AI-ingenjörer jobbar med varje dag:

### Kreditkortsbedrägerier!

Varje dag görs **miljontals** kortbetalningar världen över.  
De flesta är normala köp – men en liten del är **bedrägerier (Fraud)**:
- Tjuvar som stjäl ett kortnummer och handlar utan ägarens vetskap
- Falska köp online
- Identitetsstöld

Banker vill ha en AI som **automatiskt** kan stoppa bedrägerier i realtid –  
redan innan banken godkänner betalningen!

```
Du klickar "Betala" på nätet
          │
          ▼
AI analyserar köpet på 0.1 sekunder
          │
    ┌─────┴──────┐
    │             │
 NORMALT     BEDRÄGERI?
 ✅ Godkänd   🚨 Stoppad!
```

---
## ⚠️ Del 2 – Det stora problemet: Obalanserad data

Här kommer det knepiga:

Av alla kreditkortsköp i världen är ungefär **99.8% normala** och bara **0.2% bedrägerier**.

Det kallas **Obalanserad data (Imbalanced Data)** – en kategori är extremt mycket vanligare.

### Den bedrägliga statistiken! 🎭

Föreställ dig denna AI-modell:

```python
def min_super_ai(köp):
    return "INTE BEDRÄGERI"   # Alltid samma svar!
```

Hur bra är den här modellen? Om vi mäter **Noggrannhet (Accuracy)**:

> Av 1000 köp är 998 normala och 2 bedrägerier.  
> Om vi gissar "inte bedrägeri" på **allt** → vi har rätt 998/1000 = **99.8% noggrannhet!**

**Men modellen hittar NOLL bedrägerier! Den är helt värdelös!** 🤦

> 🎯 **Lärdom:** Hög noggrannhet (Accuracy) garanterar INTE att modellen är bra.  
> Speciellt inte när data är obalanserad!

### 💬 Reflektionsfråga 5.1

En läkare testar ett cancertest.  
I gruppen som testas har 1% cancer och 99% inte cancer.

Testet säger "ingen cancer" på alla patienter och får 99% noggrannhet.

**Fråga:** Är det ett bra test? Vad är det egentliga problemet?

**Fråga 2:** Vad är värre för en patient – att testet missar en riktig cancer,  
eller att det felaktigt flaggar en frisk person som sjuk?  
*(Det finns inget universellt rätt svar – det beror på kontext!)*

---
## 📥 Del 3 – Ladda bedrägeridatan

Vi ska använda ett riktigt dataset med kreditkortstransaktioner.  
Datan hämtas från internet – det kan ta en liten stund att ladda!

> **OBS:** Kolumnerna heter V1, V2, ..., V28 av integritetsskäl –  
> de riktiga kolumnnamnen (t.ex. "butik", "belopp", "land") är dolda för att skydda kundernas integritet.  
> Kolumnen `Class` är det vi vill förutsäga: `0` = normalt köp, `1` = bedrägeri.

In [ ]:
import pandas as pd

print("Laddar data från internet... (kan ta 30-60 sekunder)")

url = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv'
df = pd.read_csv(url)

print(f"✅ Data laddad!")
print(f"Antal transaktioner: {len(df):,}")
print(f"Antal kolumner: {len(df.columns)}")

---
## 🔍 Del 4 – Utforska obalansen

Nu ska vi se hur obalanserad datan verkligen är:

In [ ]:
# Räkna normala köp vs bedrägerier
antal_normala  = (df['Class'] == 0).sum()
antal_bedragerier = (df['Class'] == 1).sum()
totalt = len(df)

print(f"Normala transaktioner:  {antal_normala:>7,}  ({antal_normala/totalt:.2%})")
print(f"Bedrägerier:            {antal_bedragerier:>7,}  ({antal_bedragerier/totalt:.2%})")
print(f"Totalt:                 {totalt:>7,}")
print()
print(f"Det finns {antal_normala // antal_bedragerier} normala transaktioner för varje bedrägeri!")

### Visualisera obalansen

En bild säger mer än tusen ord:

In [ ]:
import matplotlib.pyplot as plt

etiketter = ['Normala köp', 'Bedrägerier']
antal = [antal_normala, antal_bedragerier]
farger = ['#4CAF50', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stapeldiagram
axes[0].bar(etiketter, antal, color=farger)
axes[0].set_title('Antal transaktioner', fontsize=13)
axes[0].set_ylabel('Antal')
for i, v in enumerate(antal):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Kakdiagram
axes[1].pie(antal, labels=etiketter, colors=farger,
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Fördelning (%)', fontsize=13)

plt.suptitle('Obalanserad data (Imbalanced Data) – kreditkortstransaktioner',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🪤 Del 5 – Noggrannhetsfällan (Accuracy Paradox)

Låt oss bevisa problemet med ett experiment.  
Vi skapar en "dum" modell som alltid svarar "Inte bedrägeri":

In [ ]:
import pandas as pd

# Den "dumma" modellen: säg alltid "Inte bedrägeri" på alla transaktioner
dummy_svar = pd.Series([0] * len(df))  # 0 = Inte bedrägeri, 1 = Bedrägeri

# Räkna hur ofta den har "rätt"
antal_ratt = (dummy_svar == df['Class']).sum()
noggrannhet_dum = antal_ratt / len(df)

print(f"Den 'dumma' modellens noggrannhet (Accuracy): {noggrannhet_dum:.2%}")
print()
print("🤔 Det ser imponerande ut – men modellen hittar NOLL bedrägerier!")
print(f"   Den missar {(df['Class'] == 1).sum():,} bedrägerier – alla!"  )
print()
print("➡️  Det är just det här problemet vi ska lösa i nästa lektion!")

### 💬 Reflektionsfråga 5.2

Du har nu sett att en "dum" modell som aldrig hittar bedrägerier ändå kan få 99,8% noggrannhet.

**Fråga 1:** Tänk dig att du är ingenjör på en bank. Du visar din chef att AI:n är 99,8% korrekt. 
Chefen är imponerad. Men du vet att den missar alla bedrägerier! 
Vad säger du till din chef?

**Fråga 2:** Vad för slags mått tror du vore bättre att använda – om man vill kontrollera 
om AI:n faktiskt *hittar* bedrägerier?
*(Tips: Tänk på en lärare som rättar prov. Vad räknar en rättvis lärare på?)*

---
> 📌 **Kom ihåg:** Noggrannhet (Accuracy) är inte alltid rätt mått. 
> I Lektion 6 tränar vi en riktig AI och ser vad som händer i verkligheten!

---
## 🎓 Bra jobbat – du är klar med Lektion 5!

### Vad du lärt dig idag:

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Obalanserad data** | En kategori är extremt mycket vanligare än en annan | Imbalanced Data |
| **Noggrannhet** | Andel rätta svar totalt | Accuracy |
| **Noggrannhetsfällan** | När Accuracy ser bra ut trots värdelös modell | Accuracy Paradox |
| **Falsk negativ** | Missad detektion (tjuven slipper undan!) | False Negative |
| **Falsk positiv** | Falskt larm (oskyldig stoppas) | False Positive |

### 👉 I Lektion 6 ska vi:
1. Dela upp datan i träning och test
2. Träna en riktig AI-modell (XGBoost)  
3. Rita ut en **Förväxlingsmatris (Confusion Matrix)** och se exakt hur många bedrägerier AI:n hittar – och missar!

*"Det svåra är inte att bygga en AI. Det svåra är att bygga en AI som faktiskt löser rätt problem."* 🤖